# RNN models: Bi-RNN vs Bi-LSTM vs Bi-GRU

Seminar 2. The three models share one architecture and differ **only in the recurrent cell**:

`fastText embedding (frozen) → Bi-{RNN|LSTM|GRU} (128 units/direction) → [max pool ; attention pool] → 6 sigmoid outputs`

Same 5 stratified folds, same `comment_heavy` input and same metrics as the TF-IDF and RoBERTa experiments, so the numbers can be compared directly.

**Runtime → Change runtime type → GPU (T4 is enough).**

In [ ]:
# 1. Get the code (push your branch to GitHub first, then set BRANCH)
BRANCH = "main"
!git clone -q -b {BRANCH} https://github.com/mnhhuyen/AI4SE_toxic_comments.git
%cd AI4SE_toxic_comments
!pip install -q -e ".[dev]" tabulate
!python -m pytest -q tests/test_recurrent_models.py 2>&1 | tail -3

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU!")

## 2. Download fastText vectors (~1 GB zip, a few minutes)

In [ ]:
!mkdir -p data/embeddings/fasttext
!wget -q -c https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip -P data/embeddings/fasttext/
!unzip -q -n data/embeddings/fasttext/wiki-news-300d-1M.vec.zip -d data/embeddings/fasttext/
!ls -lh data/embeddings/fasttext/

## 3. Clean data and write the 5 folds (same seed/split as the other experiments)

In [ ]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
from toxic_comments.cleaning import process_cleaning, validate_training_data
from toxic_comments.config import (ID_COLUMN, K_FOLD_DATA_DIR, LABEL_COLUMNS, MODELS_DIR,
                                   PROCESSED_DATA_DIR, RAW_DATA_DIR, RESULTS_DIR)
from toxic_comments.splits import save_kfold_datasets

clean_path = PROCESSED_DATA_DIR / "train_clean.csv"
if clean_path.exists():
    data = pd.read_csv(clean_path)
else:
    data = process_cleaning(validate_training_data(pd.read_csv(RAW_DATA_DIR / "train.csv")))
    clean_path.parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(clean_path, index=False)

for column in ["comment_text", "comment_light", "comment_heavy"]:
    data[column] = data[column].fillna("").astype(str)
data = data[data["is_empty_heavy"] == 0].reset_index(drop=True)

keep = [ID_COLUMN, "comment_text", "comment_light", "comment_heavy", *LABEL_COLUMNS]
save_kfold_datasets(data[keep], n_splits=5, strategy="stratified")
print(data.shape)

## 4. Train the 3 models on 5 folds

Each fold prints the vocabulary coverage by fastText and the loss per epoch. Already finished models are skipped, so you can re-run this cell after a disconnect.

In [ ]:
from toxic_comments.train import train_from_folds

RNN_MODELS = ["birnn_max_attention", "bilstm_max_attention", "bigru_max_attention"]
train_minutes = {}

for name in RNN_MODELS:
    if (RESULTS_DIR / f"{name}_fold_metrics.csv").exists():
        print(f"skip {name}: results already exist")
        continue
    print(f"
===== {name} =====")
    started = time.perf_counter()
    train_from_folds(folds_dir=K_FOLD_DATA_DIR, model_name=name, output_dir=MODELS_DIR)
    train_minutes[name] = (time.perf_counter() - started) / 60
    print(f"{name}: {train_minutes[name]:.1f} min for 5 folds")
train_minutes

## 5. Results (mean ± std over 5 folds)

To compare with the earlier models, this table also picks up any other `*_fold_metrics.csv` in `results/`. If you trained the baselines elsewhere, copy their CSVs into `results/` first.

In [ ]:
from toxic_comments.evaluation import summarize_folds

folds = pd.concat([pd.read_csv(p) for p in sorted(RESULTS_DIR.glob("*_fold_metrics.csv"))])
key = ["micro_f1", "macro_f1", "micro_roc_auc", "macro_roc_auc"]
summary = summarize_folds(folds)[key].sort_values(("macro_roc_auc", "mean"), ascending=False)
print(summary.to_markdown())

In [ ]:
per_label = pd.concat([pd.read_csv(RESULTS_DIR / f"{n}_fold_per_label.csv") for n in RNN_MODELS])
f1_table = per_label.groupby(["label", "model_name"])["f1"].mean().unstack().round(4)
auc_table = per_label.groupby(["label", "model_name"])["roc_auc"].mean().unstack().round(4)
print("### F1 per label
", f1_table.to_markdown())
print("
### ROC-AUC per label
", auc_table.to_markdown())

In [ ]:
import matplotlib.pyplot as plt

rnn_folds = folds[folds["model_name"].isin(RNN_MODELS)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["macro_roc_auc", "macro_f1"]):
    stats = rnn_folds.groupby("model_name")[metric].agg(["mean", "std"]).loc[RNN_MODELS]
    ax.bar(stats.index, stats["mean"], yerr=stats["std"], capsize=4)
    ax.set_title(f"{metric} (mean ± std, 5 folds)")
    ax.set_ylim(stats["mean"].min() - 3 * stats["std"].max() - 0.01, min(1, stats["mean"].max() + 0.01))
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "rnn_comparison.png", dpi=150)
plt.show()

## 6. Training loss curves (fold 1)

The vanilla RNN should converge more slowly / to a higher loss than LSTM and GRU: this is the vanishing-gradient argument for gated cells.

In [ ]:
import joblib

fitted = {n: joblib.load(MODELS_DIR / "fold_1" / f"{n}.joblib") for n in RNN_MODELS}
for name, model in fitted.items():
    plt.plot(range(1, len(model.history_) + 1), model.history_, marker="o", label=name)
    print(f"{name}: fastText coverage of vocab = {model.embedding_coverage_:.1%}")
plt.xlabel("epoch"); plt.ylabel("train BCE loss"); plt.legend(); plt.title("Training loss, fold 1")
plt.savefig(RESULTS_DIR / "rnn_loss_curves.png", dpi=150)
plt.show()

## 7. What does the attention look at? (qualitative analysis for the report)

In [ ]:
from toxic_comments.cleaning import clean_heavy, clean_light

examples = [
    "You are such an idiot, I hope something bad happens to you.",
    "I will find you and kill you",
    "Thanks so much for fixing this, really appreciate it!",
]
model = fitted["bigru_max_attention"]
for text in examples:
    cleaned = clean_heavy(clean_light(text))
    probs = model.predict_proba(pd.Series([cleaned]))[0]
    print(text)
    print("  probs:", dict(zip(LABEL_COLUMNS, probs.round(3))))
    print(model.token_attention(cleaned).sort_values("attention", ascending=False).head(5).to_string(index=False), "
")

## 8. Error analysis: worst mistakes on fold 1's test set

In [ ]:
test = pd.read_csv(K_FOLD_DATA_DIR / "fold_1" / "test.csv")
test["comment_heavy"] = test["comment_heavy"].fillna("").astype(str)
scores = model.predict_proba(test["comment_heavy"])
test["toxic_prob"] = scores[:, 0]
pd.set_option("display.max_colwidth", 120)
print("False negatives (toxic but low score):")
display(test[test["toxic"] == 1].nsmallest(5, "toxic_prob")[["comment_text", "toxic_prob"]])
print("False positives (clean but high score):")
display(test[test["toxic"] == 0].nlargest(5, "toxic_prob")[["comment_text", "toxic_prob"]])

## 9. Download the results (CSV + figures) for the report

In [ ]:
!zip -q -r rnn_results.zip results/
from google.colab import files
files.download("rnn_results.zip")